# ANN — Hyperparameter Optimization with Optuna
Automates search over architecture (layers, neurons) and training hyperparameters (lr, dropout, batch size, optimizer).

## Setup

In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Data Preparation

In [ ]:
df = pd.read_csv('data/fashion_mnist.csv')
X = df.iloc[:, 1:].values / 255.0
y = df.iloc[:, 0].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class FashionDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.features)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]

train_dataset = FashionDataset(X_train, y_train)
test_dataset  = FashionDataset(X_test,  y_test)

## Flexible ANN Architecture

In [ ]:
class DynamicANN(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(num_hidden_layers):
            layers += [
                nn.Linear(in_dim, neurons_per_layer),
                nn.BatchNorm1d(neurons_per_layer),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ]
            in_dim = neurons_per_layer
        layers.append(nn.Linear(in_dim, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

## Optuna Objective

In [ ]:
def objective(trial):
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 4)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 32, 256, step=32)
    dropout_rate      = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
    learning_rate     = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    weight_decay      = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    batch_size        = trial.suggest_categorical("batch_size", [32, 64, 128])
    optimizer_name    = trial.suggest_categorical("optimizer", ["Adam", "SGD", "RMSprop"])
    epochs            = trial.suggest_int("epochs", 10, 30, step=10)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, pin_memory=True)

    model = DynamicANN(784, 10, num_hidden_layers, neurons_per_layer, dropout_rate).to(device)
    criterion = nn.CrossEntropyLoss()

    optimizers = {
        "Adam":    optim.Adam,
        "SGD":     optim.SGD,
        "RMSprop": optim.RMSprop
    }
    optimizer = optimizers[optimizer_name](model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)
            _, predicted = torch.max(model(features), 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return correct / total

## Run Study

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"Best accuracy: {study.best_value:.4f}")
print(f"Best params:   {study.best_params}")

## Visualization

In [ ]:
optuna.visualization.matplotlib.plot_optimization_history(study)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

In [ ]:
optuna.visualization.matplotlib.plot_param_importances(study)
plt.tight_layout()
plt.show()

## Retrain Best Model

In [ ]:
p = study.best_params
train_loader = DataLoader(train_dataset, batch_size=p['batch_size'], shuffle=True, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=p['batch_size'], shuffle=False, pin_memory=True)

best_model = DynamicANN(784, 10, p['num_hidden_layers'], p['neurons_per_layer'], p['dropout_rate']).to(device)
criterion = nn.CrossEntropyLoss()
optimizers = {"Adam": optim.Adam, "SGD": optim.SGD, "RMSprop": optim.RMSprop}
optimizer = optimizers[p['optimizer']](best_model.parameters(), lr=p['learning_rate'], weight_decay=p['weight_decay'])

for epoch in range(p['epochs']):
    best_model.train()
    total_loss = 0
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(best_model(features), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}  Loss: {total_loss/len(train_loader):.4f}")

# Final accuracy
best_model.eval()
correct = total = 0
with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)
        _, predicted = torch.max(best_model(features), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
print(f"Final Test Accuracy: {correct/total*100:.2f}%")

torch.save(best_model.state_dict(), 'ann_fashion_mnist_optuna_best.pth')